# Formula 1 Race Strategy Simulator
## Notebook 02: Monte Carlo Uncertainty & Dynamic Programming Optimization

This notebook demonstrates **Phase 2** and **Phase 3** mathematical algorithms:
1. **Vectorized Monte Carlo Uncertainty**: Simulating lap pace stochasticity $\epsilon \sim \mathcal{N}(0, \sigma^2)$, pit stop transit variations, and random incidents.
2. **Statistical Risk Metrics**: Expectation, Standard Deviation, Value-at-Risk ($P_{95}$ VaR), and Expected Shortfall ($CVaR_{95}$).
3. **Win Probability Matrix**: Quantifying pairwise head-to-head dominance.
4. **Dynamic Programming Optimization**: Bellman backward induction across the Directed Acyclic Graph (DAG) of valid race stints.
5. **Pit Window Tolerance**: Computing the flexible pit window where strategic loss $\le \tau = 2.0\text{s}$.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import RaceModel
from src.config import BAHRAIN_CONFIG, DEFAULT_COMPOUNDS
from src.strategies import Strategy, Stint
from src.stochastic import StochasticConfig
from src.montecarlo import MonteCarloEngine, compute_win_probability_matrix
from src.optimization import StrategyOptimizer

sns.set_theme(style="darkgrid")
print("Optimization & Monte Carlo engine initialized.")

### 1. Vectorized Monte Carlo Simulation

Under race conditions, lap times vary due to tyre graining, traffic, and driver precision. Pit stops also feature execution variance:
$$T_{pit} \sim \mathcal{N}(\mu_{pit}, \sigma_{pit}^2)$$

We simulate $N = 1,000$ stochastic iterations for three candidate strategies:
1. **1-Stop (S-H)**: Soft (18) $\to$ Hard (39)
2. **2-Stop (S-M-M)**: Soft (15) $\to$ Medium (21) $\to$ Medium (21)
3. **2-Stop (S-H-S)**: Soft (14) $\to$ Hard (28) $\to$ Soft (15)

In [ ]:
model = RaceModel(BAHRAIN_CONFIG)

strategies = [
    Strategy([Stint(DEFAULT_COMPOUNDS["Soft"], 18), Stint(DEFAULT_COMPOUNDS["Hard"], 39)], name="1-Stop (S-H)"),
    Strategy([Stint(DEFAULT_COMPOUNDS["Soft"], 15), Stint(DEFAULT_COMPOUNDS["Medium"], 21), Stint(DEFAULT_COMPOUNDS["Medium"], 21)], name="2-Stop (S-M-M)"),
    Strategy([Stint(DEFAULT_COMPOUNDS["Soft"], 14), Stint(DEFAULT_COMPOUNDS["Hard"], 28), Stint(DEFAULT_COMPOUNDS["Soft"], 15)], name="2-Stop (S-H-S)"),
]

stoch_config = StochasticConfig(lap_noise_std=0.25, pit_noise_std=0.6, num_simulations=1000)
mc_engine = MonteCarloEngine(model, stoch_config, seed=42)
mc_results = mc_engine.run_simulation(strategies)

print(f"Executed {len(strategies)} strategies across {stoch_config.num_simulations} stochastic iterations.")

### 2. Risk Metrics & Distribution Analysis

Let us inspect the distribution moments and tail risks:
* **Mean & Std**: Expected total duration and volatility.
* **VaR 95%**: 95th percentile worst-case race time.
* **CVaR 95%**: Conditional expected time given that the outcome falls in the worst 5% tail.

In [ ]:
import pandas as pd

table = []
for res in mc_results:
    table.append({
        "Strategy": res.strategy_name,
        "Mean (s)": round(res.mean_time, 2),
        "Std (s)": round(res.std_dev, 2),
        "Median (s)": round(res.median_time, 2),
        "P95 VaR (s)": round(res.p95_time, 2),
        "CVaR 95 (s)": round(res.cvar_95, 2),
    })

df_metrics = pd.DataFrame(table)
print(df_metrics.to_string(index=False))

### 3. Visualizing Uncertainty: Empirical Distributions & Boxplots

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# KDE Plot
for res in mc_results:
    sns.kdeplot(res.raw_times, ax=ax1, label=res.strategy_name, fill=True, alpha=0.3, linewidth=2)

ax1.set_title("Kernel Density Estimation of Race Duration", fontweight="bold")
ax1.set_xlabel("Race Time (s)")
ax1.set_ylabel("Probability Density")
ax1.legend()

# Boxplot
data = [res.raw_times for res in mc_results]
labels = [res.strategy_name for res in mc_results]
ax2.boxplot(data, tick_labels=labels, showmeans=True)
ax2.set_title("Strategy Race Duration Boxplots & Dispersion", fontweight="bold")
ax2.set_ylabel("Total Race Time (s)")
ax2.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

### 4. Pairwise Win Probability Matrix

In head-to-head racing, the probability that strategy $A$ defeats strategy $B$ is given by:
$$P(T_A < T_B) = \frac{1}{N} \sum_{i=1}^N \mathbb{I}\big(T_A^{(i)} < T_B^{(i)}\big)$$

In [ ]:
win_matrix, strat_names = compute_win_probability_matrix(mc_results)

plt.figure(figsize=(7, 5))
sns.heatmap(win_matrix, annot=True, fmt=".1%", cmap="coolwarm", cbar=False,
            xticklabels=strat_names, yticklabels=strat_names)
plt.title("Head-to-Head Win Probability Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Opponent Strategy")
plt.ylabel("Candidate Strategy")
plt.tight_layout()
plt.show()

### 5. Dynamic Programming (Bellman Backward Induction)

The strategy optimization problem is formulated as a shortest-path problem on a Directed Acyclic Graph (DAG), where each node is a lap index $l \in \{0, 1, \dots, N\}$ and edges represent stints of compound $c$:
$$V(l) = \min_{c \in \mathcal{C}, l' \in (l, N]} \Big[ \text{LapPace}(l \to l', c) + T_{pit}(l') + V(l') \Big]$$

In [ ]:
optimizer = StrategyOptimizer(model)

opt_result = optimizer.optimize_dp(max_stops=2)

print(f"Optimal Strategy Found: {opt_result.optimal_strategy.name}")
print(f"Optimal Race Time: {opt_result.optimal_strategy.total_time:.3f} s")
print(f"Graph Nodes Evaluated: {opt_result.dp_nodes_evaluated}")
print(f"Execution Time: {opt_result.computation_time_ms:.2f} ms")

print("
Stint Breakdown:")
for i, stint in enumerate(opt_result.optimal_strategy.stints, 1):
    print(f"  Stint {i}: {stint.compound.name} for {stint.laps} laps")

### 6. Pit Stop Window Sensitivity

In real racing, pit stops cannot always happen on the exact theoretical lap due to yellow flags, overtaking opportunities, or traffic.
We calculate the **pit window tolerance** $[\text{lap}_{open}, \text{lap}_{close}]$ within which stopping incurs $\le \tau = 2.0\text{s}$ penalty.

In [ ]:
windows = optimizer.compute_pit_windows(opt_result.optimal_strategy, tolerance_seconds=2.0)

for w in windows:
    print(f"Pit Stop #{w.pit_index + 1}:")
    print(f"  Optimal Lap: Lap {w.optimal_lap}")
    print(f"  Window Range: Lap {w.window_open_lap} to Lap {w.window_close_lap} (Size: {w.window_size} laps)")